In [ ]:
import json
import re

#les 7champs que  le modèle doit produire
CHAMPS_REQUIS={"image_quality": str,"predicted_class":str , "confidence":float , "visual_evidence":str , "justification":str , "limitations" :str , "warning": str } # Changed 'warnings' to 'warning'

Classes_Valides={"pneumonie","normale","incertain"}
Qualités_Valides={"bonne","moyenne","mauvaise"}


#Ecriture fonction d'extraction


def extraire_json_brut(texte):
    texte = texte.strip()  # enlève les espaces au début et à la fin

    # Cas 1 : la réponse est déjà un JSON pur
    if texte.startswith("{"):
        return texte

    # Cas 2 : le JSON est dans un bloc ```json ... ```
    match = re.search(r"```json\s*(\{.*?\})\s*```", texte, re.DOTALL)
    if match:
        return match.group(1)

    # Cas 3 : bloc ``` sans préciser json
    match = re.search(r"```\s*(\{.*?\})\s*```", texte, re.DOTALL)
    if match:
        return match.group(1)

    # Cas 4 : JSON quelque part dans le texte
    match = re.search(r"\{.*\}", texte, re.DOTALL)
    if match:
        return match.group(0)

    # Rien trouvé
    return None


# Ecriture fonction de validation


def valider_json(data):
    erreurs = []  # liste vide, on va la remplir si on trouve des problèmes

    # Vérifier que chaque champ requis est présent et du bon type
    for champ, type_attendu in CHAMPS_REQUIS.items():
        if champ not in data:
            erreurs.append(f"Champ manquant : '{champ}'")
        elif not isinstance(data[champ], type_attendu):
            erreurs.append(f"Type incorrect pour '{champ}'")

    # Vérifier les valeurs autorisées
    if "predicted_class" in data:
        if data["predicted_class"].lower() not in Classes_Valides:
            erreurs.append(f"Classe invalide : '{data['predicted_class']}'")

    if "image_quality" in data:
        if data["image_quality"].lower() not in Qualités_Valides:
            erreurs.append(f"Qualité invalide : '{data['image_quality']}'")

    if "confidence" in data and isinstance(data["confidence"], float):
        if not (0.0 <= data["confidence"] <= 1.0):
            erreurs.append(f"Confidence hors limite : {data['confidence']}")

    return erreurs


# Ecriture du fallback

def reponse_fallback(raison):
    return {
        "image_quality":   "mauvaise",
        "predicted_class": "incertain",
        "confidence":      0.0,
        "visual_evidence": "Non disponible",
        "justification":   "Le modèle n'a pas produit de réponse exploitable.",
        "limitations":     raison,
        "warning":         "Résultat non fiable — relecture manuelle obligatoire.",
    }


# Ecriture fonction principale

def parser_reponse(reponse_modele):

    # --- 1. Extraire le JSON brut ---
    json_brut = extraire_json_brut(reponse_modele)

    if json_brut is None:
        print("[ERREUR] Aucun JSON trouvé dans la réponse.")
        return reponse_fallback("Aucun JSON détecté.")

    # --- 2. Convertir le texte JSON en dictionnaire Python ---
    try:
        data = json.loads(json_brut)
    except json.JSONDecodeError as e:
        print(f"[ERREUR] JSON malformé : {e}")
        return reponse_fallback(f"JSON malformé : {e}")

    # --- 3. Valider les champs ---
    erreurs = valider_json(data)

    if erreurs:
        print(f"[AVERTISSEMENT] {len(erreurs)} problème(s) :")
        for err in erreurs:
            print(f"  - {err}")
        data["warning"] = "Réponse partielle : " + "; ".join(erreurs)
        data.setdefault("predicted_class", "incertain")
        data.setdefault("confidence", 0.0)
    else:
        print("[OK] JSON valide.")

    return data


#  Test : Ce bloc ne s'exécute que si tu lances CE fichier directement
if __name__ == "__main__":

    print("===== TEST 1 : JSON parfait =====")
    t1 = '{"image_quality": "bonne", "predicted_class": "pneumonie", "confidence": 0.87, "visual_evidence": "Opacités basales", "justification": "Compatible pneumonie.", "limitations": "Aucune.", "warning": "Confirmer avec radiologue."}'
    print(parser_reponse(t1))

    print("\n===== TEST 2 : JSON dans du texte =====")
    t2 = 'Voici mon analyse :\n```json\n{"image_quality": "moyenne", "predicted_class": "normale", "confidence": 0.72, "visual_evidence": "Pas d\'opacité", "justification": "Poumons clairs.", "limitations": "Qualité moyenne.", "warning": "Usage pédagogique."}\n```'
    print(parser_reponse(t2))

    print("\n===== TEST 3 : Pas de JSON du tout =====")
    t3 = "Je pense que c'est peut-être une pneumonie."
    print(parser_reponse(t3))

    print("\n===== TEST 4 : Champ manquant =====")
    t4 = '{"predicted_class": "pneumonie", "confidence": 0.9}'
    print(parser_reponse(t4))

===== TEST 1 : JSON parfait =====
[AVERTISSEMENT] 1 problème(s) :
  - Champ manquant : 'warnings'
{'image_quality': 'bonne', 'predicted_class': 'pneumonie', 'confidence': 0.87, 'visual_evidence': 'Opacités basales', 'justification': 'Compatible pneumonie.', 'limitations': 'Aucune.', 'warning': "Réponse partielle : Champ manquant : 'warnings'"}

===== TEST 2 : JSON dans du texte =====
[AVERTISSEMENT] 1 problème(s) :
  - Champ manquant : 'warnings'
{'image_quality': 'moyenne', 'predicted_class': 'normale', 'confidence': 0.72, 'visual_evidence': "Pas d'opacité", 'justification': 'Poumons clairs.', 'limitations': 'Qualité moyenne.', 'warning': "Réponse partielle : Champ manquant : 'warnings'"}

===== TEST 3 : Pas de JSON du tout =====
[ERREUR] Aucun JSON trouvé dans la réponse.
{'image_quality': 'mauvaise', 'predicted_class': 'incertain', 'confidence': 0.0, 'visual_evidence': 'Non disponible', 'justification': "Le modèle n'a pas produit de réponse exploitable.", 'limitations': 'Aucun JSON 